#  Feature Engineering: Bridging Specs to Price
**Objective:** Transform raw technical specifications into meaningful market indicators to help our Decision Tree understand the 'Value' of a laptop.
**Project:** Laptop Price Prediction (ENSIA)
**Member:** Lyna

---

## 1. Setup & Data Loading
We load our cleaned dataset to begin the transformation process.

In [ ]:
import pandas as pd
import numpy as np
import os

# Locate the cleaned data
dataset_path = '../06_Model_Training_Evaluation/final_cleaned_dataset.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'final_cleaned_dataset.csv' # Fallback if running from root

df = pd.read_csv(dataset_path)
print(f" Loaded {len(df)} records with {len(df.columns)} initial columns.")

## 2. Screen Metrics Engineering (PPI & Pixels)
Pixels are a silent price driver. A 1080p screen is worth much more than a 720p screen, but the raw text '1920x1080' is hard for a model to read. We will calculate **Total Pixels** and **PPI (Pixels Per Inch)**.

In [ ]:
def extract_dimensions(res_str):
    try:
        if pd.isna(res_str) or 'x' not in str(res_str):
            return 1920, 1080
        parts = str(res_str).lower().split('x')
        return int(parts[0]), int(parts[1])
    except:
        return 1920, 1080

dims = df['SCREEN_RESOLUTION'].apply(extract_dimensions)
df['RES_WIDTH'] = [x[0] for x in dims]
df['RES_HEIGHT'] = [x[1] for x in dims]
df['TOTAL_PIXELS'] = df['RES_WIDTH'] * df['RES_HEIGHT']

# PPI (Pixels Per Inch) - Measures clarity
df['PPI'] = np.sqrt(df['RES_WIDTH']**2 + df['RES_HEIGHT']**2) / df['SCREEN_SIZE']

print(" Screen metrics calculated (PPI and Total Pixels).")

## 3. Storage Efficiency (SSD vs HDD)
In modern laptops, SSD capacity adds much more to the price than HDD capacity. We will create a weighted **Storage Score**.

In [ ]:
# SSD is 5x more impactful on price than HDD in our estimation
df['STORAGE_SCORE'] = (df['SSD_GB'] * 1.0) + (df['HDD_GB'] * 0.2)

print(" Storage Score added (Weighted SSD/HDD).")

## 4. Market Tiering (Categorical logic)
We group CPUs and Brands into Tiers based on market position (High-End vs Budget) to reduce noise.

In [ ]:
def get_cpu_tier(cpu):
    cpu = str(cpu).upper()
    if any(x in cpu for x in ['I9', 'RYZEN 9', 'M1 MAX', 'M2 MAX', 'M3 MAX']): return '4_Enthusiast'
    if any(x in cpu for x in ['I7', 'RYZEN 7', 'M1 PRO', 'M2 PRO', 'M3 PRO']): return '3_High-End'
    if any(x in cpu for x in ['I5', 'RYZEN 5', 'M1', 'M2', 'M3']): return '2_Mid-Range'
    return '1_Entry-Level'

df['CPU_TIER'] = df['CPU'].apply(get_cpu_tier)

def get_brand_tier(brand):
    brand = str(brand).upper()
    if brand in ['APPLE', 'RAZER', 'MICROSOFT']: return 'Premium'
    if brand in ['HP', 'DELL', 'LENOVO', 'ASUS', 'MSI']: return 'Mainstream'
    return 'Budget'

df['BRAND_TIER'] = df['LAPTOP_BRAND'].apply(get_brand_tier)

print(" CPU and Brand Tiers assigned.")

## 5. Identifying Gaming Machines
We check if the laptop is a dedicated gaming machine based on model names and refresh rates.

In [ ]:
def identify_gaming(row):
    model = str(row['LAPTOP_MODEL']).upper()
    if any(x in model for x in ['ROG', 'TUF', 'LEGION', 'OMEN', 'PREDATOR', 'ALIENWARE', 'VICTUS']): return 1
    if row['SCREEN_FREQUENCY_NUM'] > 60: return 1
    return 0

df['IS_GAMING'] = df.apply(identify_gaming, axis=1)

print(" Gaming indicator created.")

## 6. Saving Final Results
We save the enriched dataset to its final location for model training.

In [ ]:
output_path = '../06_Model_Training_Evaluation/final_cleaned_dataset.csv'
df.to_csv(output_path, index=False)

print(f" Dataset Enriched! Final column count: {len(df.columns)}")
print("Top 5 records preview:")
df[['LAPTOP_BRAND', 'CPU_TIER', 'STORAGE_SCORE', 'PPI', 'IS_GAMING', 'PRICE']].head()